# 🎓 第7章 第1课：风险模型与组合优化

**本节目标**：用简化的风格风险模型估计协方差，再做最小方差/最大 Sharpe 组合。

对应原书第7章收益率模型、Barra 风险模型、组合优化部分。

> ⚠️ 本仓库没有行业分类/Barra 因子数据；这里用 7 个风格因子做一个**迷你风险模型**，
> 演示“因子暴露 → 协方差 → 组合优化”的流程。数值不代表真实 A 股结论。

In [1]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src' / 'factor_zero').exists():
            return p
    raise FileNotFoundError('找不到仓库根目录，请从仓库内启动 notebook')
ROOT = _find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from factor_zero.data import load_ashare_panel
from factor_zero.factors import build_factor_returns
from factor_zero.anomalies import rolling_ivol
from factor_zero.portfolio import (
    FACTOR_ORDER, build_style_exposures, factor_covariance,
    specific_variance, portfolio_risk, min_variance_weights,
    max_sharpe_weights, style_regression, smart_beta_weights,
)
from factor_zero.statistics import factor_summary, nw_t_stat

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
print('✅ 环境就绪，开始学习！')

✅ 环境就绪，开始学习！


## 1) 迷你风险模型：暴露 × 因子协方差 + 特质风险

```
Σ = B Σ_f B' + D
```

- **B**：股票的风格暴露（截面 z-score）；
- **Σ_f**：因子协方差（滚动 36 个月，年化）；
- **D**：特质方差（滚动回归残差波动率的平方）。

这就是 Barra 风险模型的“降维”思想：不直接估计 N×N 协方差，而是用 K 个因子 + 特质风险。

In [2]:
panel = load_ashare_panel()
ret = panel['ret']
factors = build_factor_returns(panel, n_groups=10)
beta = ret.rolling(24).cov(panel['market']).div(panel['market'].rolling(24).var(), axis=0)
ivol = rolling_ivol(ret, factors[FACTOR_ORDER], window=36)
date = ret.index[-1]
B = build_style_exposures(panel, date, beta)
fcov = factor_covariance(factors, date, window=36)
sv = specific_variance(ivol, date, B.index)
print('暴露矩阵:', B.shape, '| 因子协方差:', fcov.shape, '| 特质方差:', sv.shape)
B.head(3).round(3)

暴露矩阵: (81, 7) | 因子协方差: (7, 7) | 特质方差: (81,)


,MKT,SMB,HML,WML,RMW,CMA,TURN
000001,0.290,-0.211,2.241,0.393,-0.118,1.365,0.361
000002,0.877,1.563,3.291,-1.396,-4.192,-0.216,-1.243
000063,1.011,0.125,-0.355,0.257,-0.444,0.361,-3.550


In [3]:
fmean = factors[FACTOR_ORDER].loc[:date].tail(36).mean() * 12  # 年化因子溢价
mu = B @ fmean  # 用因子模型给股票预期收益
Sigma = B @ fcov @ B.T + pd.DataFrame(np.diag(sv), index=B.index, columns=B.index)
print('预期收益（年化）:', mu.describe().round(3).to_dict())
print('协方差矩阵:', Sigma.shape)
pd.Series(np.diag(Sigma), index=B.index).describe().round(3)

预期收益（年化）: {'count': 81.0, 'mean': -0.0, 'std': 0.719, 'min': -3.318, '25%': -0.333, '50%': 0.186, '75%': 0.395, 'max': 1.973}
协方差矩阵: (81, 81)


count    81.000
mean      0.755
std       1.012
min       0.053
25%       0.213
50%       0.511
75%       0.784
max       7.270
dtype: float64

## 2) 三种组合：等权 / 最小方差 / 最大 Sharpe

用 `scipy.optimize` 做长仓、权重上限 10% 的约束优化。

In [4]:
w_eq = pd.Series(1.0 / len(B), index=B.index)
w_mv = min_variance_weights(Sigma, max_weight=0.10)
w_ms = max_sharpe_weights(mu, Sigma, max_weight=0.10)
rows = {}
for name, w in [('等权', w_eq), ('最小方差', w_mv), ('最大Sharpe', w_ms)]:
    r = portfolio_risk(w, B, fcov, sv)
    rows[name] = {
        '预期收益': float(w @ mu),
        '总波动': r['total_vol'],
        '因子波动': r['factor_vol'],
        '特质波动': r['specific_vol'],
        'Sharpe': float((w @ mu) / r['total_vol']) if r['total_vol'] > 0 else np.nan,
    }
pd.DataFrame(rows).T.round(4)

,预期收益,总波动,因子波动,特质波动,Sharpe
等权,-0.0000,0.0251,0.0000,0.0251,-0.0000
最小方差,-0.0004,0.0217,0.0005,0.0217,-0.0188
最大Sharpe,0.1692,0.0814,0.0746,0.0324,2.0801


In [5]:
for name, w in [('等权', w_eq), ('最小方差', w_mv), ('最大Sharpe', w_ms)]:
    print(name, w.sort_values(ascending=False).head(5).round(3).to_dict())

等权 {'000001': 0.012, '000002': 0.012, '000063': 0.012, '000100': 0.012, '000333': 0.012}
最小方差 {'600887': 0.034, '000651': 0.029, '600999': 0.029, '600438': 0.024, '600519': 0.024}
最大Sharpe {'600999': 0.084, '000001': 0.051, '000651': 0.049, '000776': 0.047, '600887': 0.046}


## 3) 风险分解：风险从哪里来？

对最小方差组合做风险归因：每个因子贡献多少波动，特质风险贡献多少。
贡献之和 = 总波动（Euler 分解）。

In [6]:
r = portfolio_risk(w_mv, B, fcov, sv)
contrib = r['factor_contrib'].copy()
contrib['特质'] = r['specific_contrib']
print(f'总波动: {r["total_vol"]*100:.2f}%  因子: {r["factor_vol"]*100:.2f}%  特质: {r["specific_vol"]*100:.2f}%')
(contrib * 100).round(2)

总波动: 2.17%  因子: 0.05%  特质: 2.17%


MKT     0.00
SMB     0.00
HML    -0.00
WML    -0.00
RMW     0.00
CMA    -0.00
TURN    0.00
特质      2.17
dtype: float64

## ✋ 试一试

1. 把权重上限从 10% 改成 5%，最小方差组合的波动怎么变？
2. 把因子协方差窗口从 36 个月改成 24 个月，结果稳定吗？
3. 如果把 `mu` 换成历史均值，最大 Sharpe 组合会怎么变？（提示：模型错位）

## ✅ 本节小结

- 风险模型用 **B Σ_f B' + D** 降维估计协方差；
- 最小方差组合只依赖协方差；最大 Sharpe 还依赖预期收益；
- 预期收益估计误差会放大组合权重（模型错位）；
- 风险归因把总波动拆成因子贡献 + 特质贡献。

> 下一课：Smart Beta 与因子择时。